# Solutions · Chapter 02-04 · Missing values, duplicates, and impossible things

Worked answers with reasoning. E4 and E14 are the pair to read carefully: one shows the median
surviving what destroys the mean, the other prices every handling choice in the currency that
matters - the model's error.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(9)
n_customers = 3000
income = np.round(rng.lognormal(10.3, 0.5, n_customers), -2)
p_decline = 0.05 + 0.55 * (income > np.quantile(income, 0.70))
declined = rng.random(n_customers) < p_decline
observed_income = np.where(declined, np.nan, income)
print(f"{declined.mean():.1%} declined | true mean {income.mean():,.0f} | "
      f"observed mean {np.nanmean(observed_income):,.0f}")

## E1 · The three mechanisms

**One question:** *does the chance of being missing depend on the value that is missing?*

- **MCAR** - it depends on nothing. A form lost in the post.
- **MAR** - it depends only on things you *did* record. App users skip the field, and you know who
  is an app user.
- **MNAR** - it depends on the missing value itself. High earners decline to state their income.

**`dropna()` is safe only under MCAR.** Under MAR the loss is repairable using the recorded columns;
under MNAR nothing in the data can repair it, because the information needed is exactly what is
absent.

**And you cannot distinguish MAR from MNAR by looking at the data**, since that would require the
values you do not have. It is a judgement about the collection process.

## E2 · Why mean imputation changed nothing

The mean of the filled column is a weighted average of the observed values and the value you
inserted. If the value you insert *is* the observed mean, the average cannot move: you are adding
copies of the current average to itself.

Formally, with `k` observed values averaging `m` and `p` gaps filled with `m`, the new mean is
`(k*m + p*m) / (k + p) = m`. Exactly, always, for any dataset.

**So mean imputation cannot correct a biased complete-case mean - it can only preserve it**, while
making the column look complete. Three further harms, in order of how often they matter:

1. **Variance shrinks.** A fifth of the column is now one identical number, so the standard
   deviation falls and every correlation involving it is attenuated toward zero.
2. **A spike appears** at exactly one value, which trees split on and histograms make obvious.
3. **The fact of missingness is deleted** - and under MNAR that fact was informative.

**When is mean imputation acceptable?** When missingness is genuinely MCAR and you need a complete
matrix for an algorithm that cannot handle nulls. Even then, add the indicator column - it costs one
column and can only help.

## E3 · Why a sentinel is worse than a null

A null is **honest**: it says "no value here", every tool recognises it, `isna()` counts it, and
aggregations either skip it or propagate `NaN` loudly.

A sentinel is a **lie in the type system**. `-1` is a valid float. It passes every check, loads
without warning, appears in `describe()` as an ordinary minimum, and participates in arithmetic,
sorting and comparisons as though it were a measurement.

The specific damage is that it has a **position on the scale**, and that position is arbitrary. In
the chapter, `-1` for "never rented" sits below `0` for "rented today", so a recency filter selects
the least engaged customers as the most engaged. A different sentinel - `9999` - would have put them
at the other end and produced the opposite error.

**The general statement:** a null is outside the ordering; a sentinel is inside it, in a place
nobody chose for a reason.

## E4 · Ten incomes, three refusals

In [ ]:
incomes = np.array([20, 22, 25, 28, 30, 34, 40, 55, 80, 120], dtype=float)
observed = incomes[:7]                                   # the top three decline

filled = np.concatenate([observed, np.repeat(observed.mean(), 3)])
print(f"true mean           : {incomes.mean():.2f}k")
print(f"complete-case mean  : {observed.mean():.2f}k  ({observed.mean() / incomes.mean() - 1:+.1%})")
print(f"mean-imputed mean   : {filled.mean():.2f}k  (identical, by construction)")
print()
print(f"true median         : {np.median(incomes):.2f}k")
print(f"complete-case median: {np.median(observed):.2f}k  "
      f"({np.median(observed) / np.median(incomes) - 1:+.1%})")

| | True | Complete-case | Error |
|---|---|---|---|
| Mean | 45.40k | 28.43k | **-37.4%** |
| Median | 32.00k | 28.00k | **-12.5%** |

**The median held up much better**, and the reason is worth having precisely.

The mean is a sum: removing the three largest values removes 255 of the 454 total - more than half
the mass - so it collapses. The median is a *position*: removing three values from the top end
shifts the middle by a few positions, and the values near the middle are close together, so the
number barely moves.

**But do not over-learn this.** The median is more *robust*, not more *correct*. It is still 12.5%
too low, and it would be equally destroyed if the missing values had come from the middle rather
than the top. Robustness buys you tolerance to a particular pattern of damage - here, damage
concentrated in a tail - and nothing more.

**And the deeper point:** neither summary is right, because both are computed from a sample that is
missing its top 30% by construction. Choosing a more robust statistic manages the symptom. The
disease is that a third of the population is absent, and no statistic fixes that.

## E5 · The sentinel's arithmetic

In [ ]:
share_sentinel, real_mean, share_real_low = 0.25, 30.0, 0.10

reported_mean = share_sentinel * (-1) + (1 - share_sentinel) * real_mean
selected = share_sentinel + (1 - share_sentinel) * share_real_low
should_select = (1 - share_sentinel) * share_real_low

print(f"reported mean of the column : {reported_mean:.2f}  (real values average {real_mean})")
print(f"filter 'value <= 5' selects : {selected:.1%}")
print(f"it should select            : {should_select:.1%}")
print(f"of those selected, {1 - should_select / selected:.0%} are sentinels, not measurements")

- **Reported mean: 22.25**, against a real average of 30. The sentinels drag it down by 7.75 - a
  quarter of the true level - and the number looks entirely plausible.
- **The filter selects 32.5% of rows and should select 7.5%.** More than four times too many, and
  **77% of everyone it selects is a sentinel** - rows where the quantity was never measured at all.

That last figure is the one to carry. The filter does not merely over-select; the group it produces
is *mostly* composed of the cases it was meant to exclude. Anything computed about that group - a
conversion rate, an average spend, a campaign response - describes people who have no value in this
field, not people with a low one.

**And notice the two harms are different in kind.** The mean is *diluted*, which is a bias you could
in principle correct if you knew the sentinel share. The filter is *inverted*, which no correction
to the mean would ever reveal. Aggregate checks do not catch threshold bugs.

## E6 · A quality report

In [ ]:
def quality_report(frame):
    rows = []
    for name in frame.columns:
        col = frame[name]
        is_text = pd.api.types.is_string_dtype(col) or col.dtype == object
        top = col.value_counts(dropna=True)
        rows.append({
            "column": name,
            "dtype": str(col.dtype),
            "pct_missing": round(100 * col.isna().mean(), 1),
            "distinct": col.nunique(),
            "min": col.min() if pd.api.types.is_numeric_dtype(col) else "",
            "max": col.max() if pd.api.types.is_numeric_dtype(col) else "",
            "top_value": top.index[0] if len(top) else "",
            "top_share": round(100 * top.iloc[0] / len(col), 1) if len(top) else 0.0,
            "would_merge": bool(is_text and col.str.strip().str.lower().nunique() < col.nunique()),
        })
    return pd.DataFrame(rows).set_index("column")


demo = pd.DataFrame({
    "customer_id": [1, 2, 2, 3, 3, 4, 5, 6, 7],
    "station": ["north", "North", "North", "north ", "south", "south", "SOUTH", "east", "east"],
    "age": [34, 41, 41, 29, 29, 200, 38, -5, 52],
    "income": [np.nan, 30000, 30000, np.nan, 41000, np.nan, 52000, 28000, np.nan],
})
quality_report(demo)

Everything wrong with that table is visible in one output: `station` flagged `would_merge`, `age`
with a minimum of -5 and a maximum of 200, `income` 44% missing, and `customer_id` with a top value
appearing twice when it should be unique.

**Two design choices worth copying:**

- **`top_share` is more useful than `top_value` alone.** A continuous column whose most common value
  covers 25% of rows is a sentinel; the value itself is only interesting once the share has flagged
  it.
- **It returns a DataFrame rather than printing.** So it can be sorted, filtered, saved and *diffed
  between two files* - which turns "look at the data" into a check you can run on every delivery, as
  01-04's E11 argued.

**What it deliberately does not do:** decide anything. It reports; the handling decision needs a
human who knows why the values are missing. A tool that auto-cleaned these columns would silently
make four consequential choices.

## E7 · Finding sentinels

In [ ]:
def find_sentinels(series, plausible=None, max_share=0.02):
    """Flag values that repeat implausibly often, or fall outside a plausible range."""
    flags = []
    shares = series.value_counts(normalize=True)
    for value, share in shares.items():
        if share > max_share:
            flags.append((value, f"{share:.1%} of rows - too repetitive for a continuous column"))
    if plausible is not None:
        low, high = plausible
        for value in sorted(series[(series < low) | (series > high)].unique()):
            flags.append((value, f"outside the plausible range {low} to {high}"))
    return pd.DataFrame(flags, columns=["value", "reason"])


days = rng.integers(0, 60, n_customers).astype(float)
never_rented = rng.random(n_customers) < 0.18
recorded_days = pd.Series(np.where(never_rented, -1.0, days))

print(find_sentinels(recorded_days, plausible=(0, 365)).to_string(index=False))

`-1` is flagged twice - by both tests independently - which is the ideal outcome: two unrelated
diagnostics agreeing is much stronger evidence than either alone, exactly as the two changepoint
scans agreeing was in 02-02.

**Two honest limitations:**

- **A sentinel inside the plausible range is invisible to the second test.** `0` meaning "not
  recorded" in a column where 0 is a legitimate value can only be found by the repetition test - or
  by asking.
- **The repetition threshold is a guess.** Set it too low and every integer column with few distinct
  values is flagged; too high and a rare sentinel slips through. It is a *pointer to something to
  look at*, not a decision.

**And the check no code can perform:** whether `-1` means "never rented", "unknown", or "the system
was down". Those three demand different handling - a separate category, an imputation, and a row
exclusion respectively - and the column cannot tell you which.

## E8 · A column that is 40% missing

**Three actions:**

1. **Drop the column.** If 40% is missing and the field is weakly related to the target, a
   feature available for only 60% of rows adds complexity and a production dependency for little
   gain.
2. **Keep it, impute, and add an indicator.** If it is a strong predictor where present, 60% of a
   valuable signal beats none - and the indicator lets the model use "we do not know this" as
   information in its own right.
3. **Go and get the values, or change the collection.** If the field is central to the decision, 40%
   missing is a data-collection problem being handed to a modelling team, and fixing it upstream
   dominates every analytical workaround.

**The deciding fact: *why* is it missing, and will it still be missing at prediction time?**

That second half is the one people forget. A field that is 40% missing historically but always
present for new records is a different problem from one that will be 40% missing in production
forever. And if it is missing *because* of something related to the outcome - only unprofitable
customers lack a credit score - then it is MNAR, imputation is guesswork, and the indicator column
is carrying most of the real signal.

**One more check before deciding:** is the missingness concentrated in time? A field that is 100%
missing before 2022 and 0% after is not a 40%-missing column; it is two different datasets stapled
together, and 02-02 has the tools for it.

## E9 · Development drops, production fills with zero

**Two separate problems.**

**Problem 1: the development estimate is optimistic.** Dropping rows with missing values trains and
evaluates on complete cases only. If missingness is related to anything - and it usually is - the
complete cases are an easier, cleaner, more predictable subset. The reported accuracy describes that
subset, not the population the model will actually meet.

**Problem 2, and this is the worse one: training and serving disagree.** In development the model
never saw a missing value. In production it receives `0` in place of every one - and `0` is not a
neutral placeholder, it is a specific point on the scale. An income of 0, a `days_since_last_rental`
of 0, a temperature of 0: each is a strong claim, and the model responds to it as if it were
measured. Every row with a missing field gets a confidently wrong prediction, and the pattern of
which rows are affected is systematic.

**Which is worse: the second, clearly.** Problem 1 makes the number wrong; problem 2 makes the
*predictions* wrong, for a specific and probably important subgroup, silently. And it degrades as
missingness rises - so it gets worse exactly when the upstream system is having a bad day.

**This is training/serving skew** (13-04), and the structural fix is that missing-value handling
must live *inside the fitted pipeline*, not in a notebook cell and separately in the serving code.
One object, fitted on training data, applied identically in both places. That is what 04-07's
`Pipeline` is for, and this is the failure that motivates it.

## E10 · "How do you handle missing data?"

> First I find out why it is missing, because that decides everything else. If the chance of a value
> being absent has nothing to do with the value itself, dropping the rows costs precision and
> nothing else. If it depends on something I recorded, I can impute from those columns. If it depends
> on the missing value itself - high earners declining to state income - then no imputation recovers
> it and I should say so rather than produce a confident wrong average. In practice I almost always
> add a "was missing" indicator column, because the fact of absence is often informative and every
> other approach deletes it. I check whether the rows with missing values differ from the rest on
> what I can see. And I make sure the handling lives inside the fitted pipeline, so production does
> not fill with zeros what development dropped.

**What is being assessed:** whether you treat missingness as a question about the world or as a
preprocessing step. Listing five imputation methods answers a different question - the interviewer
wants to know whether you would notice that `dropna()` changed who is in your dataset.

## E11 · Salary, optional for two years then mandatory

**What I would check first: whether the missingness is concentrated in the first two years.** One
`groupby` on year of the missing rate answers it, and it changes the diagnosis completely.

**If it is concentrated:** this is not one column with 30% missing. It is a column that did not
exist for the first period, and the two periods are different datasets. That makes the mechanism
close to MCAR *within* the early period - whoever filled it in optionally may still have
self-selected, so not quite - but the dominant pattern is a policy change, not a property of the
people. Options: restrict the analysis to the mandatory period and say so, or model the two periods
separately, or use the early data only for questions that do not involve salary.

**If it is not concentrated** - if the rate is a steady 30% throughout - then the "optional field"
story does not explain it, and I would go back to the collection team, because there is a second
mechanism nobody has mentioned.

**Either way I would compare the two groups** on everything known for both: tenure, role, region,
department. If people who left it blank differ systematically, that is evidence about the mechanism
even without seeing their salaries.

**What I would tell the person who asked for "average salary":**

> "Among employees with a salary recorded, the average is X. Salary was optional before 2023, so
> roughly a third of records lack it, and those are concentrated in the earlier period. If you need a
> figure for the whole workforce, the honest one is from 2023 onwards, which is Y. I can also give a
> range under different assumptions about the missing group if the decision needs it."

Two numbers and a caveat, rather than one number and a footnote nobody reads.

## E12 · The hospital columns

| Column | Likely mechanism or defect | The check | The handling |
|---|---|---|---|
| `blood_pressure`, 15% missing | Probably **MAR or MNAR** - it is measured when clinicians think it matters, so absence correlates with how well the patient looked | Missing rate by ward, admission type and severity. If it varies strongly, it is informative | Impute from correlated observations **and** keep an indicator - "not measured" is a clinical decision and a real signal |
| `smoker` with `Y`, `N`, `y`, `Yes`, `unknown`, blank | **Messy categories plus two kinds of missing** | `value_counts(dropna=False)`; then `.str.strip().str.upper()` | Map to three explicit states: yes, no, **not recorded**. `unknown` and blank may differ - one was asked, the other was not - so check before merging them |
| `weight_kg`, range 0 to 700 | **Sentinels and unit errors at both ends** | Histogram plus `value_counts()` at the extremes | 0 is a sentinel, not a weight - set it to missing. 700 needs a human: plausible for a very few patients, and also what 70.0 looks like after a decimal-point error |

**The `smoker` column carries the most important lesson.** Collapsing `unknown` and blank is the
obvious tidy-up and it may destroy a real distinction: `unknown` usually means *asked and not
answered*, blank means *never asked*. Those describe different patients and different clinical
pathways. **Tidying categories is a modelling decision, not housekeeping** - merge only labels that
mean the same thing, and check with someone who knows.

**And the 700 kg row is the chapter's central judgement in miniature:** an impossible value is an
error, an extreme value may be the most important patient in the file, and only domain knowledge
separates them. That is the next chapter.

## E13 · Explaining it to Maria

> The people who left the income box blank were mostly your better-off customers - they are the ones
> who would rather not say. So when I deleted those rows, I deleted the top end, and the average came
> out about a seventh too low. Instead I kept everyone, filled the blanks with a typical value, and
> added a column recording who had left it blank - because that turned out to be worth knowing on
> its own.

(69 words.)

**Why the last clause is the one that matters:** it reframes the blank from a nuisance to be
disposed of into a finding. Maria now knows something she did not know before - that her wealthier
customers decline to answer - which is more useful to her than the corrected average.

## E14 · What each choice costs, in the model's error

In [ ]:
spending = 0.05 * income + rng.normal(0, 300, n_customers)          # SYNTHETIC
train, test = slice(0, 2200), slice(2200, None)
y_test = spending[test]

median_train = np.nanmedian(observed_income[train])
X_test_imputed = np.where(np.isnan(observed_income[test]), median_train,
                          observed_income[test]).reshape(-1, 1)

def evaluate(X_tr, y_tr, X_te, label):
    model = LinearRegression().fit(X_tr, y_tr)
    print(f"  {label:<36} MAE {mean_absolute_error(y_test, model.predict(X_te)):7.1f} EUR")

evaluate(income[train].reshape(-1, 1), spending[train], income[test].reshape(-1, 1),
         "oracle: true income everywhere")

complete = ~np.isnan(observed_income[train])
evaluate(observed_income[train][complete].reshape(-1, 1), spending[train][complete], X_test_imputed,
         "drop rows with missing income")

X_train_imputed = np.where(np.isnan(observed_income[train]), median_train,
                           observed_income[train]).reshape(-1, 1)
evaluate(X_train_imputed, spending[train], X_test_imputed, "median-impute, no indicator")

with_flag = lambda X, raw: np.column_stack([X.ravel(), np.isnan(raw).astype(float)])
evaluate(with_flag(X_train_imputed, observed_income[train]), spending[train],
         with_flag(X_test_imputed, observed_income[test]), "median-impute + missing indicator")

print(f"  {'baseline: always the mean spend':<36} MAE "
      f"{mean_absolute_error(y_test, [spending[train].mean()] * len(y_test)):7.1f} EUR")

| Treatment | MAE | |
|---|---|---|
| Oracle - true income everywhere | **233.0** | the best any model could do |
| Drop rows with missing income | 471.5 | |
| Median-impute, no indicator | 489.7 | **worse than dropping** |
| Median-impute **+ indicator** | **342.4** | |
| Baseline - always predict the mean | 746.3 | |

**The indicator column is worth 147 MAE**, recovering about 57% of the gap between naive imputation
and the impossible oracle. One extra column of zeros and ones.

**Why it works so well here.** Under MNAR, "declined to answer" is itself a strong predictor of high
income - the two groups differ by more than 20,000 EUR - so the flag carries much of the information
the imputation destroyed. The model can learn "when income is the median *and* the flag is set,
predict higher", which is precisely the correction the imputed value needed.

**Why plain median imputation is *worse than dropping rows*.** Dropping at least trains on honest
pairs of income and spending. Imputing inserts several hundred training rows that assert a false income
and pair it with a genuinely high spend, actively teaching the model a wrong relationship near the
median.
Deleting data is bad; inventing it is worse.

**And read the baseline row before drawing a conclusion.** Every treatment beats "always predict the
mean" comfortably, so all four are useful models. The question was never whether to handle the
missing values, but which handling loses least - and the answer is the one that throws nothing away.

**The transferable rule:** *keep the fact, fill the value.* It costs one column, cannot hurt, and
here it was worth more than the choice between every other option.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **02-05 · Distributions, outliers,
and transformations** - where the values are right, and surprising.